<a href="https://colab.research.google.com/github/farrelrassya/python-for-finance/blob/main/ch03_data_types_and_structures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 3 — Data Types and Structures

> *"Bad programmers worry about the code. Good programmers worry about data structures and their relationships."*  
> -- Linus Torvalds

This notebook accompanies **Chapter 3** of *Python for Finance* (Yves Hilpisch, 2nd edition). The chapter is foundational: every model, pipeline, and analysis we will write in later chapters is ultimately built from the primitives introduced here.

We approach the material from a **machine-learning / data-science perspective**. The Python primitives -- `int`, `float`, `bool`, `str`, `tuple`, `list`, `dict`, `set` -- are not just abstract programming concepts. They are the building blocks of:

- **Tensors and arrays** (a NumPy `ndarray` is, at its core, a contiguous block of `float`s with a `tuple` shape).
- **Hyperparameter configurations** (`dict` is the *de facto* config container in scikit-learn, PyTorch, and Hugging Face).
- **Vocabularies and label spaces** (`set` is the natural representation for unique-element collections in NLP and classification).
- **Training histories and batches** (`list` of metric values; `tuple` of `(features, label)` pairs).
- **Text preprocessing** (`str` methods and regular expressions are the workhorse of NLP feature engineering).

The chapter covers the following types:

| Object type | Meaning | Used for |
|---|---|---|
| `int` | Integer value | Natural numbers, indices, counts |
| `float` | Floating-point number | Real numbers, weights, losses |
| `bool` | Boolean value | Truth values, masks |
| `str` | String object | Character, word, text |
| `tuple` | Immutable container | Fixed-size records, shapes |
| `list` | Mutable container | Changing collections |
| `dict` | Mutable container | Key-value store |
| `set` | Mutable container | Unique-element collections |

Python is a **dynamically typed** language: the interpreter infers the type of an object at runtime. This contrasts with statically typed languages like C, where types must be declared before compile time. Dynamic typing is convenient for exploratory data analysis but has performance implications we will revisit in Chapter 4 when we introduce NumPy.

## Setup

This chapter uses only the Python standard library (`decimal`, `keyword`, `re`, `datetime`, `random`). No third-party packages are required, which makes this notebook fully runnable on Google Colab without `pip install` commands.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Stdlib imports used throughout this chapter
import decimal
from decimal import Decimal
import keyword
import re
from datetime import datetime
from random import randint, seed

print("Setup complete -- all modules from the Python standard library.")

Setup complete -- all modules from the Python standard library.


All five modules are part of CPython itself, with no external dependencies. From an **ML/DS engineering perspective** this is the most reliable kind of code: standard library modules ship with every Python installation, are version-stable for years, and have zero supply-chain risk -- a property that matters enormously when deploying models behind firewalls or in regulated environments.

## 3.1 Basic Data Types

### 3.1.1 Integers

The `int` type represents whole numbers. In Python, integers are first-class objects with built-in methods -- a consequence of the language's design principle that **everything in Python is an object**.

In [ ]:
a = 10
type(a)

int

The built-in `type()` function returns the class of an object. This is the workhorse of debugging type errors in ML pipelines, where a silent type mismatch (an `int` where a `float` was expected, or a Python list where a tensor was expected) can produce wrong gradients without any error message.

**Production insight:** in deployed inference services we frequently log `type(input)` at the API boundary. A model trained on `float32` tensors will silently mis-predict if served `int` inputs because of implicit casting in the framework's type promotion rules.

In [ ]:
a.bit_length()

4

The integer $a = 10$ requires **4 bits** to represent in binary, since $10 = 1010_2$ and $\lceil \log_2(10+1) \rceil = 4$. More precisely:

$$\text{bit\_length}(n) = \lceil \log_2(|n| + 1) \rceil \quad \text{for } n \neq 0$$

**Why this matters in ML:** when designing categorical embeddings, we often need to know how many bits are required to index a vocabulary. A vocabulary of $50{,}000$ tokens needs $\lceil \log_2(50{,}001) \rceil = 16$ bits per index, which is why most modern tokenizers use `int32` or `int64` token IDs even though `int16` would suffice -- alignment with hardware word size is faster than space-optimal packing.

In [ ]:
a = 100000
a.bit_length()

17

Now $a = 100{,}000$ requires **17 bits**, since $2^{16} = 65{,}536 < 100{,}000 < 131{,}072 = 2^{17}$. The bit count grows logarithmically with the magnitude of the number -- a property that underlies the $O(\log n)$ complexity of many integer algorithms.

In [ ]:
googol = 10 ** 100
googol

10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

Python represents the **googol** ($10^{100}$) exactly. Unlike C's fixed-width `int32` (which overflows at $2^{31} - 1 \approx 2.15 \times 10^9$) or `int64` (which caps at $\approx 9.22 \times 10^{18}$), **Python's `int` is arbitrary precision**: the interpreter allocates as many bytes as needed.

> **Large Integers.** Python integers can be arbitrarily large. The interpreter simply uses as many bits/bytes as needed to represent the numbers.

**ML/DS implication:** this is why factorials, combinatorial counts, and exact probability ratios in pure Python never overflow. However, this same feature *destroys* NumPy's vectorization speed if Python big-ints leak into a NumPy pipeline, because NumPy must downcast to fixed-width types or fall back to Python `object` arrays (which are 10-100x slower).

In [ ]:
googol.bit_length()

333

The googol requires **333 bits**, which agrees closely with the formula $\log_2(10^{100}) = 100 \cdot \log_2(10) \approx 332.19$, rounded up to $333$.

A 333-bit integer occupies roughly $\lceil 333 / 8 \rceil = 42$ bytes -- compared to 8 bytes for a `numpy.int64`. The 5x storage overhead is the price of arbitrary precision.

In [ ]:
1 + 4

5

Basic arithmetic on integers returns an integer. Division behaves differently:

In [ ]:
1 / 4

0.25

Note the value `0.25` -- not an `int`. Let's check the type explicitly:

In [ ]:
type(1 / 4)

float

Notice the type coercion: `1 / 4` returns `0.25`, a `float`, **even though both operands are `int`**. This is Python 3's *true division*: the `/` operator always returns a `float`, while `//` performs integer (floor) division.

$$\underbrace{1}_{\text{int}} \;\; / \;\; \underbrace{4}_{\text{int}} \;\; = \;\; \underbrace{0.25}_{\text{float}}$$

**Strategic insight:** this is one of the most common sources of silent bugs when porting Python 2 code (where `/` between two `int`s did floor division). For ML code, the modern Python 3 semantics are exactly what we want -- losses, learning rates, and gradients are inherently real-valued.

### 3.1.2 Floats

The `float` type represents real numbers using the IEEE 754 double-precision format -- the same 64-bit representation used by NumPy's default `float64`, MATLAB's `double`, and most ML frameworks' "high-precision" mode.

In [ ]:
1.6 / 4

0.4

And the type of this result:

In [ ]:
type(1.6 / 4)

float

So far, so clean. But float arithmetic hides a fundamental issue.

In [ ]:
b = 0.35
type(b)

float

Now an arithmetic operation that should produce a clean rational:

In [ ]:
b + 0.1

0.44999999999999996

We expected $0.35 + 0.10 = 0.45$, but Python returns **`0.44999999999999996`**. This is not a Python bug -- it is a fundamental limitation of binary floating-point representation.

A real number $0 < n < 1$ is internally encoded as a finite binary expansion:

$$n = \frac{x_1}{2} + \frac{x_2}{4} + \frac{x_3}{8} + \dots = \sum_{i=1}^{52} \frac{x_i}{2^i} \quad \text{with } x_i \in \{0, 1\}$$

(plus an exponent and sign bit, totalling 64 bits in IEEE 754 double precision). Decimal numbers like $0.35$ have **infinite repeating binary expansions** -- analogous to how $\frac{1}{3} = 0.\overline{3}$ never terminates in decimal. Truncation to 52 mantissa bits introduces the $\approx 4 \times 10^{-17}$ error visible above.

**This is the single most important fact in numerical ML.** It is why:

- We never compare floats with `==`. Use `abs(a - b) < eps` or `numpy.isclose`.
- Loss values that "should be zero" hover at $10^{-8}$ to $10^{-15}$ at convergence.
- Mixed-precision training (`float16`) requires loss scaling to avoid underflow.
- Cross-entropy is computed via `log_softmax` rather than `log(softmax(x))` -- the latter loses precision for confident predictions.

In [ ]:
c = 0.5
c.as_integer_ratio()

(1, 2)

The number $0.5$ is stored **exactly**, because it has the finite binary representation $0.5 = \frac{1}{2} = 0.1_2$. The `as_integer_ratio()` method returns the exact rational form $\frac{1}{2}$.

Other exactly representable numbers include $0.25, 0.125, 0.75$ -- in general, any number of the form $k / 2^m$ for integers $k, m$.

In [ ]:
b.as_integer_ratio()

(3152519739159347, 9007199254740992)

For $b = 0.35$, Python reveals what is actually stored:

$$b = \frac{3{,}152{,}519{,}739{,}159{,}347}{9{,}007{,}199{,}254{,}740{,}992} = \frac{3{,}152{,}519{,}739{,}159{,}347}{2^{53}}$$

The denominator is exactly $2^{53} = 9{,}007{,}199{,}254{,}740{,}992$, the size of the mantissa space in IEEE 754 double precision. This ratio differs from the true $\frac{7}{20} = 0.35$ by approximately $4 \times 10^{-17}$ -- the *machine epsilon* for doubles.

**Quantitative insight:** IEEE 754 double precision provides $\log_{10}(2^{52}) \approx 15.95$ significant decimal digits. This is the upper bound on the precision of *any* arithmetic in standard NumPy, PyTorch, or TensorFlow without explicit `float128` or `Decimal` types.

#### Arbitrary precision with `decimal`

When the 15-digit precision of `float` is insufficient -- as in financial accounting, where summing millions of transactions must reconcile to the cent -- Python provides the `decimal` module.

In [ ]:
decimal.getcontext()

Context(prec=28, rounding=ROUND_HALF_EVEN, Emin=-999999, Emax=999999, capitals=1, clamp=0, flags=[], traps=[InvalidOperation, DivisionByZero, Overflow])

The default `Decimal` context provides **28 digits of precision** -- nearly double the 16 digits of `float64`. Other settings include:

- **`rounding=ROUND_HALF_EVEN`** (banker's rounding): rounds to the nearest even digit on ties. This matches the IEEE 754 default and is the standard in finance because it eliminates the systematic upward bias of `ROUND_HALF_UP`.
- **`traps`**: by default, invalid operations and division-by-zero raise exceptions rather than silently returning `NaN`/`Inf`. This is the *opposite* of NumPy's default behavior -- a deliberate choice for safety-critical applications.

**ML connection:** in ranking and recommender systems, `ROUND_HALF_EVEN` is preferred for tie-breaking score thresholds. Banker's rounding produces empirically more balanced positive/negative classifications than `round-half-up` over millions of decisions.

In [ ]:
d = Decimal(1) / Decimal(11)
d

Decimal('0.09090909090909090909090909091')

$\frac{1}{11}$ has the infinite repeating decimal $0.\overline{09}$. With the default precision of 28 digits, `Decimal` stores 28 digits of $09$ (with a final $1$ from rounding) -- 13 more digits than `float` would retain. Comparing:

| Type | Stored value | Error vs $\frac{1}{11}$ |
|---|---|---|
| `float` | $0.09090909090909091$ (≈17 chars displayed) | $\sim 10^{-17}$ |
| `Decimal` (prec=28) | $0.09090909090909090909090909091$ | $\sim 10^{-29}$ |

In [ ]:
decimal.getcontext().prec = 4
e = Decimal(1) / Decimal(11)
e

Decimal('0.09091')

Setting `prec = 4` gives only **4 significant digits**: $0.09091$. This is much coarser than `float` and shows that `Decimal` is configurable in *both* directions -- higher precision than `float` for finance, or lower precision for memory-constrained simulations.

In [ ]:
decimal.getcontext().prec = 50
f = Decimal(1) / Decimal(11)
f

Decimal('0.090909090909090909090909090909090909090909090909091')

With `prec = 50` we now store **50 significant digits** -- well beyond what any hardware float type can offer. The cost is speed: `Decimal` arithmetic is implemented in pure Python (since Python 3.3, partly in C) and runs **roughly 100x slower** than native `float`.

**Strategic insight for ML:** for *training* a neural network, `float32` is fast and accurate enough -- the noise from minibatch sampling dwarfs the rounding error. But for **financial backtesting**, P&L accumulation, or any pipeline where errors compound over millions of operations, `Decimal` is non-negotiable. A relative error of $10^{-15}$ per trade, accumulated over $10^9$ trades, is $10^{-6}$ -- millions of dollars on a billion-dollar book.

In [ ]:
g = d + e + f
g

Decimal('0.27272818181818181818181818181909090909090909090909')

Three `Decimal` objects with **different precisions** were summed: 28-digit `d`, 4-digit `e`, and 50-digit `f`. The result `g` carries the highest precision in scope (50 digits), but the value is "polluted" near the 5th digit by the low-precision `e`:

$$g = d + e + f \approx 3 \cdot \frac{1}{11} = 0.\overline{27}$$

The output begins $0.27272818...$ -- the deviation $0.\overline{27} - 0.27272818 \approx 1.5 \times 10^{-7}$ is exactly the propagated 4-digit error from `e`.

**Lesson:** when mixing precisions, the *lowest* precision dominates the result. This is the exact same principle as in mixed-precision deep learning: a single `float16` operation in an otherwise `float32` graph can degrade convergence.

In [ ]:
decimal.getcontext().prec = 28
print("Reset to default 28-digit precision.")

Reset to default 28-digit precision.


We reset the precision to the 28-digit default, restoring a clean state. **Always restore global state after modification** -- this idiom appears constantly in ML code that touches `numpy.random.seed`, `torch.set_default_dtype`, or matplotlib's `rcParams`.

### 3.1.3 Booleans

The `bool` type has exactly two values: `True` and `False`. They are *keywords* -- reserved identifiers that cannot be reassigned.

In [ ]:
keyword.kwlist

['False',
 'None',
 'True',
 'and',
 'as',
 'assert',
 'async',
 'await',
 'break',
 'class',
 'continue',
 'def',
 'del',
 'elif',
 'else',
 'except',
 'finally',
 'for',
 'from',
 'global',
 'if',
 'import',
 'in',
 'is',
 'lambda',
 'nonlocal',
 'not',
 'or',
 'pass',
 'raise',
 'return',
 'try',
 'while',
 'with',
 'yield']

Python 3 has exactly **35 keywords**. Among them are `True`, `False`, and `None` -- which behave as Boolean and null literals. Two of these (`async`, `await`) were introduced in Python 3.5 for asynchronous programming, and three (`nonlocal`, `True`, `False`) became reserved in Python 3.0.

**Practical note:** none of these names can be used as variable names. A common ML beginner mistake is `True = 1` or `lambda = 0.01` -- both are syntax errors. The conventional ML notation `lambda` for an L2 penalty is therefore typically rendered as `lam`, `lmbda`, or `weight_decay` in code.

In [ ]:
4 > 3

True

And the type of the comparison result:

In [ ]:
type(4 > 3)

bool

A **comparison expression** evaluates to a `bool`. Comparisons are the foundation of:

- **Boolean masks** in NumPy/Pandas: `arr[arr > 0]` filters in $O(n)$ time using vectorized comparisons.
- **Decision tree splits**: every internal node is a comparison `x[j] <= threshold`.
- **Early stopping**: `if val_loss > best_val_loss: ...`.

In [ ]:
int(True), int(False)

(1, 0)

The same numeric identification holds for floats:

In [ ]:
float(True), float(False)

(1.0, 0.0)

Numerically, `True` is **1** and `False` is **0**. This identification is fundamental:

$$\mathbb{1}[A] = \begin{cases} 1 & \text{if } A \text{ is true} \\ 0 & \text{otherwise} \end{cases}$$

The function $\mathbb{1}[\cdot]$ is the **indicator function**, which underlies:

- **0/1 loss**: $L_{0/1}(y, \hat{y}) = \mathbb{1}[y \neq \hat{y}]$.
- **Accuracy**: $\text{acc} = \frac{1}{n} \sum_{i=1}^{n} \mathbb{1}[y_i = \hat{y}_i]$, computed in NumPy as `(y == y_hat).mean()` -- the `.mean()` of a Boolean array silently casts `True` to 1.
- **Precision/recall/F1**: all expressible as sums of indicator functions over the confusion matrix.

In [ ]:
bool(0), bool(0.0), bool(1), bool(10.5), bool(-2)

(False, False, True, True, True)

The reverse cast -- numbers to `bool` -- follows the rule **"zero is false, everything else is true"**. Note that **`-2` casts to `True`**: the cast checks for nonzero, not for positivity.

**Pitfall:** `bool(np.array([0, 1]))` raises `ValueError` because NumPy refuses ambiguous truthiness on arrays. Code like `if my_array:` is one of the most common bugs when transitioning from pure-Python to NumPy. The fix is `if my_array.any():` or `if my_array.all():`.

### 3.1.4 Strings

The `str` type represents text. From an ML/DS standpoint, `str` is the entry point for **all NLP work**: tokenization, normalization, regular expressions, encoding/decoding all start here.

In [ ]:
t = 'this is a string object'
t.capitalize()

'This is a string object'

Splitting the string into tokens -- the foundation of every NLP pipeline:

In [ ]:
t.split()

['this', 'is', 'a', 'string', 'object']

`split()` returns a `list` of **5 tokens**. With no argument, it splits on any whitespace run -- a crude but surprisingly effective baseline tokenizer for English. More precisely, the algorithm is:

$$\text{split}(s) = [\text{maximal nonspace substrings of } s]$$

This is exactly the tokenization used in `sklearn.feature_extraction.text.CountVectorizer` when `token_pattern=None` -- and it's why scikit-learn's bag-of-words baseline is so easy to set up: the entire pipeline is just `.lower()` + `.split()` + `Counter`.

In [ ]:
t.find('string')

10

`find('string')` returns **10**, the zero-based index where the substring `'string'` starts in `t`. We can verify: `t[10:16] = 'string'`. Internally, `find()` runs the **Boyer-Moore-Horspool** algorithm in $O(n)$ average time.

**ML application:** locating named entities in raw text, sliding windows for sequence labeling, and extracting features like "does this review contain the word *amazing*?" all rely on `find()` or its case-insensitive cousin `re.search()`.

In [ ]:
t.find('Python')

-1

When the substring is absent, `find()` returns **`-1`**. This is a *sentinel value* convention -- in contrast, the related `index()` method raises `ValueError`.

**Defensive pattern:** `if t.find(needle) != -1:` is more concise than `try / except`, and the `-1` sentinel is unambiguous because string indices are always non-negative when a match exists.

In [ ]:
t.replace(' ', '|')

'this|is|a|string|object'

`replace(old, new)` swaps every occurrence of `old` with `new`. Note that **strings are immutable**: this returns a *new* string, leaving `t` unchanged. The same idiom underlies CSV/TSV separator conversion, currency symbol stripping (`'$1,234.56'.replace('$', '').replace(',', '')`), and regex-light text cleaning.

In [ ]:
'http://www.python.org'.strip('htp:/')

'www.python.org'

`strip(chars)` removes leading and trailing characters that appear in the *character set* `chars` (not the literal substring). Each of `h`, `t`, `p`, `:`, `/` is independently eligible for removal from the ends. The algorithm processes character-by-character from each end until it hits a character not in the set:

$$\texttt{'http://www.python.org'} \xrightarrow{\text{strip 'htp:/'}} \texttt{'www.python.org'}$$

**Common bug:** `'hello world'.strip('hello')` returns `' world'`, not `' world'` -- it removes the *characters* `'h', 'e', 'l', 'o'` from the ends, not the literal word `'hello'`. For literal-substring removal, use `removeprefix()` and `removesuffix()` (Python 3.9+).

#### Excursion: Printing and string replacements

In ML pipelines, formatted output appears in **logging frameworks**, **TensorBoard scalars**, and **tqdm progress bars**. Mastering `str` formatting is essential for readable training logs.

In [ ]:
print('Python for Finance')
print(t)
i = 0
while i < 4:
    print(i)
    i += 1

Python for Finance
this is a string object
0
1
2
3


The `print()` function is the most common debugging and logging primitive. Each call writes its arguments separated by spaces (the `sep=` parameter) and ends with a newline (the `end=` parameter). The implicit `\n` is what makes each iteration appear on its own line.

**Production note:** in real ML pipelines, replace `print()` with the `logging` module immediately. `logging` provides:

- Severity levels (`DEBUG`, `INFO`, `WARNING`, `ERROR`).
- Configurable destinations (file, stderr, network).
- Per-module log filters.
- Timestamps and PID stamps for distributed training.

In [ ]:
i = 0
while i < 4:
    print(i, end='|')
    i += 1

0|1|2|3|

By overriding `end='|'`, all four numbers print on **a single line** separated by `|`. This is the printf-style "no newline" trick -- exactly the mechanism behind `tqdm`'s in-place progress bars (which use `\r` carriage returns to overwrite the same line).

**Old-style `%`-formatting** (still found in legacy code and log strings):

In [ ]:
print('this is an integer %d' % 15)
print('this is an integer %4d' % 15)
print('this is an integer %04d' % 15)
print('this is a float %f' % 15.3456)
print('this is a float %.2f' % 15.3456)
print('this is a float %8.2f' % 15.3456)
print('this is a float %08.2f' % 15.3456)
print('this is a string %s' % 'Python')
print('this is a string %10s' % 'Python')

this is an integer 15
this is an integer   15
this is an integer 0015
this is a float 15.345600
this is a float 15.35
this is a float    15.35
this is a float 00015.35
this is a string Python
this is a string     Python


The format specifier follows the pattern `%[flags][width][.precision]type`:

| Specifier | Meaning | Example output |
|---|---|---|
| `%d` | Integer, no padding | `15` |
| `%4d` | Integer, width 4 (right-aligned) | `&nbsp;&nbsp;15` |
| `%04d` | Integer, width 4, **zero-padded** | `0015` |
| `%f` | Float, default 6 decimals | `15.345600` |
| `%.2f` | Float, 2 decimals | `15.35` |
| `%8.2f` | Float, width 8, 2 decimals | `&nbsp;&nbsp;&nbsp;15.35` |
| `%08.2f` | Float, width 8, 2 decimals, zero-padded | `00015.35` |
| `%s` | String, no padding | `Python` |
| `%10s` | String, width 10 (right-aligned) | `&nbsp;&nbsp;&nbsp;&nbsp;Python` |

**ML logging convention:** `'epoch %3d | loss %.4f | lr %.2e' % (epoch, loss, lr)` produces aligned columns that tail nicely in a terminal:

```
epoch   1 | loss 2.3456 | lr 1.00e-03
epoch   2 | loss 1.0234 | lr 1.00e-03
```

The fixed widths matter -- without them, columns drift as numbers grow or shrink in digit count.

**New-style `.format()` and f-strings** (preferred in modern code):

In [ ]:
print('this is an integer {:d}'.format(15))
print('this is an integer {:04d}'.format(15))
print('this is a float {:.2f}'.format(15.3456))
print('this is a float {:08.2f}'.format(15.3456))
print('this is a string {:10s}'.format('Python'))

this is an integer 15
this is an integer 0015
this is a float 15.35
this is a float 00015.35
this is a string Python    


The `.format()` syntax replaces `%` with `{}` and uses a colon as the spec separator. It is more flexible (named arguments, attribute access, nested calls) than `%`-formatting.

Notice one subtle difference: with `'{:10s}'.format('Python')` strings are **left-aligned by default** (`'Python    '`), while `'%10s' % 'Python'` right-aligns (`'    Python'`).

Since Python 3.6, **f-strings** offer the most concise syntax of all:

```python
loss, lr = 1.234, 0.001
f'epoch {epoch:3d} | loss {loss:.4f} | lr {lr:.2e}'
```

f-strings are the recommended style for all new code. They are roughly 2x faster than `%`-formatting and 1.5x faster than `.format()` in microbenchmarks because the format string is parsed at compile time.

#### Excursion: Regular expressions

Regular expressions are a domain-specific language for pattern matching. In ML/DS, they are the most common tool for **structured text extraction** -- pulling timestamps from logs, prices from product descriptions, or entity mentions from raw documents.

In [ ]:
series = '''
'01/18/2014 13:00:00', 100, '1st';
'01/18/2014 13:30:00', 110, '2nd';
'01/18/2014 14:00:00', 120, '3rd'
'''
dt = re.compile(r"'[0-9/:\s]+'")
result = dt.findall(series)
result

["'01/18/2014 13:00:00'", "'01/18/2014 13:30:00'", "'01/18/2014 14:00:00'"]

The pattern `r"'[0-9/:\s]+'"` decomposes as:

| Component | Meaning |
|---|---|
| `'` | Literal single quote |
| `[0-9/:\s]` | A character class matching any digit, slash, colon, or whitespace |
| `+` | One or more of the preceding |
| `'` | Closing literal single quote |

The `r` prefix marks the string as a **raw string**, so `\s` is not interpreted as the escape sequence for whitespace by Python *before* being passed to the regex engine -- it stays as the two-character sequence `\s` and is then interpreted as "whitespace" by the regex engine itself. **Always use raw strings for regex patterns.**

`findall()` returns **all 3 non-overlapping matches** as a `list` of strings. The pre-compiled `re.compile(...)` pattern is reusable: in a streaming pipeline that processes millions of records, compiling once and matching repeatedly is 5-10x faster than calling `re.findall(pattern, text)` each time (which re-compiles internally with a small cache).

**Production lesson:** at scale, regex patterns are themselves a maintenance liability. The pattern above does not validate that the date is a real date (`'02/30/2014'` would match), nor that the time is sensible. For production data ingestion, prefer dedicated parsers (`dateutil.parser`, `pandas.to_datetime`) which handle ambiguity, locale, and edge cases.

In [ ]:
pydt = datetime.strptime(result[0].replace("'", ''), '%m/%d/%Y %H:%M:%S')
pydt

datetime.datetime(2014, 1, 18, 13, 0)

And the formatted-string and type representations:

In [ ]:
print(pydt)
print(type(pydt))

2014-01-18 13:00:00
<class 'datetime.datetime'>


Once the timestamp string is extracted, `datetime.strptime` parses it into a structured `datetime` object. The format string `'%m/%d/%Y %H:%M:%S'` mirrors the structure of the input:

| Directive | Meaning |
|---|---|
| `%m` | 2-digit month (01-12) |
| `%d` | 2-digit day (01-31) |
| `%Y` | 4-digit year |
| `%H` | 2-digit hour (00-23) |
| `%M` | 2-digit minute (00-59) |
| `%S` | 2-digit second (00-59) |

The resulting `datetime` object supports arithmetic (`pydt2 - pydt1` yields a `timedelta`), comparisons (`pydt1 < pydt2`), and formatting back to strings (`pydt.strftime(...)`).

**ML connection:** time series ML pipelines typically convert raw timestamp strings to `datetime` (or `numpy.datetime64`, which is 10x faster but less feature-rich), then derive cyclical features:

$$\text{hour}_{\sin} = \sin\!\left(\frac{2\pi \cdot \text{hour}}{24}\right), \quad \text{hour}_{\cos} = \cos\!\left(\frac{2\pi \cdot \text{hour}}{24}\right)$$

These continuous, periodic encodings let linear models capture daily seasonality without the discontinuity of a raw "hour-of-day" integer feature.

## 3.2 Basic Data Structures

We now move from atomic types to the **container types** that hold collections of objects. Python provides four built-in collections with distinct trade-offs:

| Type | Mutable? | Ordered? | Indexable? | Unique elements? | Typical ML use |
|---|---|---|---|---|---|
| `tuple` | No | Yes | Yes (int) | No | Fixed-shape records, return values |
| `list` | Yes | Yes | Yes (int) | No | Mini-batches, training history |
| `dict` | Yes | Insertion-ordered (3.7+) | Yes (key) | Keys yes, values no | Hyperparameters, feature stores |
| `set` | Yes | No | No | Yes | Vocabularies, label spaces |

### 3.2.1 Tuples

A `tuple` is an **immutable, ordered, indexed** sequence. Once created, its contents cannot be changed.

In [ ]:
tt = (1, 2.5, 'data')
type(tt)

tuple

A tuple may contain elements of **mixed types** (here `int`, `float`, `str`). This contrasts with NumPy arrays, which enforce a single `dtype` for the entire array. The flexibility comes at a cost: tuple element access is $O(1)$ but each element is a Python object with ~28 bytes of overhead, versus 8 bytes for a `float64` in a NumPy array.

In [ ]:
tt = 1, 2.5, 'data'
type(tt)

tuple

**Parentheses are optional**: comma-separated values create a tuple regardless. This enables the elegant **multiple-return-value** idiom that is ubiquitous in ML code:

```python
def split_xy(df):
    return df.drop('y', axis=1), df['y']

X, y = split_xy(df)        # tuple unpacked into two variables
```

Behind the scenes, `return X, y` builds a 2-tuple, and `X, y = ...` unpacks it. The `train_test_split` function returns a 4-tuple, and we unpack with `X_train, X_test, y_train, y_test = train_test_split(...)`.

In [ ]:
tt[2]

'data'

And its type:

In [ ]:
type(tt[2])

str

Indexing uses **zero-based positions**. The third element `'data'` is at index `2`. Python's zero-based indexing aligns with C, NumPy, PyTorch, TensorFlow, and JAX -- but differs from MATLAB, R, and Julia (which are 1-based). This is the single most common indexing bug when porting code between ecosystems.

> **Zero-Based Numbering.** In contrast to some other programming languages like Matlab, Python uses zero-based numbering schemes. For example, the first element of a tuple object has index value 0.

In [ ]:
tt.count('data')

1

And the index of the first occurrence of `1`:

In [ ]:
tt.index(1)

0

Tuples expose only **two methods**: `count(x)` (number of occurrences) and `index(x)` (position of first occurrence). The minimal API reflects the immutability contract: there is nothing to add, remove, or sort.

**Why tuples matter in ML:**

- **Hashability**: tuples (when their contents are hashable) can be used as `dict` keys and `set` elements -- crucial for caching: `cache[(model_name, batch_size, seed)] = result`.
- **Shape descriptors**: `array.shape` returns a tuple `(n_samples, n_features)`. Shapes are immutable for safety -- you can't accidentally mutate `arr.shape` to corrupt the array.
- **Defensive programming**: returning a tuple instead of a list signals to the caller "this is a fixed record, not a collection you should append to."


### 3.2.2 Lists

A `list` is a **mutable, ordered, indexed** sequence -- the workhorse container of Python.

In [ ]:
l = [1, 2.5, 'data']
l[2]

'data'

Lists can also be constructed from any other iterable using `list()`:

In [ ]:
l = list(tt)
l

[1, 2.5, 'data']

The `list()` constructor coerces any iterable -- tuple, string, generator, NumPy array -- into a list. This is the standard way to materialize a generator's output for plotting or printing.

In [ ]:
l.append([4, 3])
l

[1, 2.5, 'data', [4, 3]]

`append(x)` adds **`x` itself** as a single element. The list `[4, 3]` is appended *as a list*, not flattened: the result has length **4**, with the fourth element being the nested list `[4, 3]`.

**Performance:** `append()` is **amortized $O(1)$**. CPython's list is a dynamic array that doubles its capacity when full, so the average cost over many appends is constant. This is why building a list via repeated `.append()` in a loop is the canonical Python idiom -- and why it's safe to do, even for lists of millions of elements.

In [ ]:
l.extend([1.0, 1.5, 2.0])
l

[1, 2.5, 'data', [4, 3], 1.0, 1.5, 2.0]

`extend(iterable)` is the **flat** counterpart: each element of the argument is appended individually. The list grew from 4 to 7 elements.

The distinction matters: `lst.append([1,2,3])` adds 1 element (a sublist), while `lst.extend([1,2,3])` adds 3 elements. In ML pipelines that accumulate per-batch metrics, `extend()` is the right choice when you have a list of new metrics to add; `append()` is right when you want to record the entire batch as one nested record.

In [ ]:
l.insert(1, 'insert')
l

[1, 'insert', 2.5, 'data', [4, 3], 1.0, 1.5, 2.0]

`insert(i, x)` places `x` *before* index `i`, shifting all subsequent elements right by one. Cost is **$O(n)$** because the underlying array must be rewritten from index `i` onward.

**Production rule of thumb:** never `insert(0, x)` in a loop -- you create an $O(n^2)$ algorithm. For prepending in a hot loop, use `collections.deque`, which has $O(1)$ insertion at both ends.

In [ ]:
l.remove('data')
l

[1, 'insert', 2.5, [4, 3], 1.0, 1.5, 2.0]

`remove(x)` deletes the **first** occurrence of `x` (raising `ValueError` if absent). Cost is $O(n)$ because the algorithm must (a) linearly scan to find `x`, then (b) shift subsequent elements left.

In [ ]:
p = l.pop(3)
print(l, p)

[1, 'insert', 2.5, 1.0, 1.5, 2.0] [4, 3]


`pop(i)` **removes and returns** the element at index `i`. The popped element here is `[4, 3]`. Without an argument, `pop()` removes the *last* element in $O(1)$ -- making `list` an efficient stack. With an index, it is $O(n)$ for the same shifting reason.

The `(removed_list, popped_element)` printout shows both halves of the operation in one line.

In [ ]:
l[2:5]

[2.5, 1.0, 1.5]

**Slicing** uses the syntax `lst[start:stop:step]`, where:

- `start` is **inclusive** (default `0`).
- `stop` is **exclusive** (default `len(lst)`).
- `step` is the stride (default `1`; negative reverses).

The slice `l[2:5]` returns elements at indices 2, 3, 4 -- which are `2.5, 1.0, 1.5`. The "stop is exclusive" convention -- a recurring Python pattern -- means that `len(l[a:b]) == b - a` whenever the slice is in-bounds. This is why `range(0, n)` produces $n$ values, not $n+1$.

**ML application:** slicing is the canonical way to create train/test splits without copying data: `X_train, X_test = X[:n_train], X[n_train:]`. NumPy and PyTorch slices are even more powerful -- they return *views* into the same memory rather than copies, which is essential when working with multi-gigabyte tensors.

#### Excursion: Control structures

Loops and conditionals are introduced after lists because **iteration in Python is fundamentally over collections**, not over numeric counters.

In [ ]:
for element in l[2:5]:
    print(element ** 2)

6.25
1.0
2.25


The `for` loop iterates over **each element** of the slice `[2.5, 1.0, 1.5]`, squaring it. The output `6.25, 1.0, 2.25` corresponds to $2.5^2, 1.0^2, 1.5^2$.

Notice we never write `for i in range(len(l)):` -- we iterate directly over elements. This is the **Pythonic** style and runs ~30% faster than index-based iteration because it avoids the per-iteration `lst[i]` lookup.

**Indentation is syntax in Python.** The 4-space indent is not cosmetic -- it defines the loop body. PEP 8 mandates 4 spaces; tabs are forbidden in mixed contexts since Python 3.

In [ ]:
r = range(0, 8, 1)
r

range(0, 8)

And its type:

In [ ]:
type(r)

range

`range(start, stop, step)` does **not** allocate a list. It returns a *lazy* `range` object that computes values on demand. Memory cost is **constant** regardless of length:

```python
range(0, 10**9)   # 48 bytes, instant
list(range(0, 10**9))   # ~30 GB, OOM
```

This is one of the most important changes from Python 2 (where `range` did allocate). In ML pipelines, lazy iteration is everywhere: PyTorch `DataLoader`, TensorFlow `tf.data.Dataset`, generator expressions -- all built on the same lazy-evaluation principle.

In [ ]:
for i in range(2, 5):
    print(l[i] ** 2)

6.25
1.0
2.25


This is the **counter-based** equivalent of the previous loop. Output is identical because we square the same elements. The Pythonic style (iterating over the slice directly) is preferred unless the index `i` is needed in the loop body -- in which case `enumerate(l)` provides both index and element together.

In [ ]:
for i in range(1, 10):
    if i % 2 == 0:
        print('%d is even' % i)
    elif i % 3 == 0:
        print('%d is multiple of 3' % i)
    else:
        print('%d is odd' % i)

1 is odd
2 is even
3 is multiple of 3
4 is even
5 is odd
6 is even
7 is odd
8 is even
9 is multiple of 3


The `if / elif / else` chain demonstrates **first-match semantics**: each integer hits exactly one branch. Note that **6 is classified as "even"**, not "multiple of 3", because the `%2 == 0` test fires first.

The modulo operator `%` returns the remainder. For integers, $a \bmod b$ has the property $0 \le a \bmod b < b$ (for $b > 0$). It is the foundation of:

- **Hash table indexing**: `hash(key) % n_buckets`.
- **Cyclic feature engineering**: `day_of_year % 7` for weekly seasonality.
- **Mini-batch sampling**: `batch_idx = (batch_idx + 1) % n_batches` to wrap around.

In [ ]:
total = 0
while total < 100:
    total += 1
print(total)

100


The `while` loop runs as long as its condition is true. After incrementing 100 times, `total` becomes **100**, which violates `total < 100`, terminating the loop.

`while` loops are preferred over `for` when:

- The number of iterations is **not known in advance** (training until convergence).
- The exit condition depends on **runtime state** (early stopping when validation loss plateaus).

**Caveat:** `while True: ... break` is a common ML pattern (training loops), but it requires careful break conditions to avoid infinite loops. Always include a maximum-iteration safeguard.

In [ ]:
m = [i ** 2 for i in range(5)]
m

[0, 1, 4, 9, 16]

**List comprehensions** are Python's most distinctive iteration construct: `[expression for item in iterable]`. The output `[0, 1, 4, 9, 16]` is the squares $0^2, 1^2, 2^2, 3^2, 4^2$.

The general form is:

$$[\,f(x) \;:\; x \in S, \; P(x)\,]$$

implemented as `[f(x) for x in S if P(x)]`. This is the imperative-syntax equivalent of mathematical set-builder notation.

**Performance:** list comprehensions are ~30% faster than equivalent `for` loops with `.append()`, because the bytecode compiler inlines the append into a specialized opcode. They are also more readable -- a single line conveys "I am transforming every element".

**ML connection:** list comprehensions are a *gateway drug* to vectorization. The line `[x**2 for x in arr]` mentally maps directly to NumPy's `arr ** 2`, which executes the same operation 100x faster in C. We will exploit this isomorphism heavily in Chapter 4.

#### Excursion: Functional programming

Python supports functional-programming idioms via `map`, `filter`, `reduce`, and anonymous (`lambda`) functions. While list comprehensions cover most use cases, these tools shine when composing pipelines.

In [ ]:
def f(x):
    return x ** 2

f(2)

4

A **named function** is defined with `def`. The body is the indented block; `return` produces the output. Functions are first-class objects in Python -- they can be assigned to variables, passed as arguments, and returned from other functions. This is the foundation of decorators, callbacks, and the entire Keras / scikit-learn API design.

In [ ]:
def even(x):
    return x % 2 == 0

even(3)

False

A **predicate** is a function that returns a `bool`. Here `even(3)` returns `False` because $3 \bmod 2 = 1 \neq 0$. Predicates are the input language for filtering operations -- both in pure Python (`filter`) and in dataframes (`df[df['x'].apply(even)]`).

In [ ]:
list(map(even, range(10)))

[True, False, True, False, True, False, True, False, True, False]

`map(f, iterable)` applies `f` to every element, returning a lazy iterator (we wrap with `list()` to materialize). The output `[True, False, True, ...]` reflects the parity pattern of `0, 1, 2, ..., 9`: 5 evens and 5 odds.

`map` is the functional analog of NumPy's `np.vectorize` -- both let you apply a scalar function elementwise. NumPy's version is *much* faster for numerical work, but Python's `map` works on any object type (strings, custom classes, etc.).

In [ ]:
list(map(lambda x: x ** 2, range(10)))

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

A **`lambda`** is an anonymous function -- syntactic sugar for a one-expression `def`. The form `lambda x: x ** 2` is equivalent to:

```python
def _(x): return x ** 2
```

Lambdas are powerful when used with `map`, `filter`, `sorted(key=...)`, and `pandas.apply`. The output is the squares $0, 1, 4, 9, 16, 25, 36, 49, 64, 81$ -- ten perfect squares.

**Caveat:** lambdas are limited to a single expression and cannot contain statements (no `if/else` blocks, no loops). For anything more complex, use a named `def`. Style guides (PEP 8, Google) generally recommend lambdas only for trivial transforms.

In [ ]:
list(filter(even, range(15)))

[0, 2, 4, 6, 8, 10, 12, 14]

`filter(pred, iterable)` keeps only the elements for which `pred(x)` is true. From `range(15)` (which is $\{0, 1, ..., 14\}$), the **8 even numbers** $\{0, 2, 4, 6, 8, 10, 12, 14\}$ are retained.

**The `map` / `filter` / `reduce` triad** mirrors the **MapReduce** paradigm of distributed computing (Hadoop, Spark). The same conceptual model -- transform, filter, aggregate -- scales from in-memory Python lists to terabyte-scale Spark RDDs. Mastering it locally builds intuition for distributed data engineering.

> **List Comprehensions, Functional Programming, Anonymous Functions.** It can be considered good practice to avoid loops on the Python level as far as possible. List comprehensions and functional programming tools like `filter()`, `map()`, and `reduce()` provide means to write code without (explicit) loops that is both compact and in general more readable.

### 3.2.3 Dictionaries

A `dict` is a **mutable, hashed key-value store**. It is by far the most-used data structure in modern ML code -- hyperparameter configs, model state dictionaries, JSON payloads, and feature stores are all dicts.

In [ ]:
d = {
    'Name': 'Angela Merkel',
    'Country': 'Germany',
    'Profession': 'Chancelor',
    'Age': 64
}
type(d)

dict

Dictionaries are constructed with **curly braces** and `key: value` pairs. Keys must be **hashable** (immutable -- `str`, `int`, `tuple` of hashables; *not* `list` or `dict`). Values can be anything.

**ML pattern:** scikit-learn estimators use `**kwargs` constructors that accept dict-style configs:

```python
config = {'C': 0.01, 'penalty': 'l2', 'max_iter': 1000}
model = LogisticRegression(**config)
```

The `**` unpacking operator splats a dict into keyword arguments. This makes config-driven training trivial: load JSON/YAML → dict → unpack into model.

In [ ]:
print(d['Name'], d['Age'])

Angela Merkel 64


Square-bracket lookup: `d[key]` retrieves the value. Average lookup cost is **$O(1)$** -- the defining feature of hash tables. CPython's dict implementation uses open addressing with quadratic probing, and (since 3.6) maintains insertion order, which is now part of the language spec (3.7+).

In [ ]:
d.keys()

dict_keys(['Name', 'Country', 'Profession', 'Age'])

Just the values:

In [ ]:
d.values()

dict_values(['Angela Merkel', 'Germany', 'Chancelor', 64])

And key-value pairs together as tuples:

In [ ]:
d.items()

dict_items([('Name', 'Angela Merkel'), ('Country', 'Germany'), ('Profession', 'Chancelor'), ('Age', 64)])

The three view objects -- `dict_keys`, `dict_values`, `dict_items` -- are **dynamic views** into the dictionary, not copies. They are iterable, support `len()` and `in`, and reflect *live* changes to the underlying dict. `items()` returns `(key, value)` tuples, which is the canonical input to the dict-iteration idiom:

```python
for key, value in d.items():
    ...
```

**Memory note:** views are $O(1)$ to construct; `list(d.keys())` materializes a real list of 4 elements (here) and is $O(n)$. Use `list(...)` only when you need a snapshot or random access.

In [ ]:
birthday = True
if birthday:
    d['Age'] += 1
print(d['Age'])

65


**In-place mutation.** `d['Age'] += 1` updates the value at key `'Age'` from 64 to **65** without copying the dict. The conditional `if birthday:` gates the mutation -- a pattern that appears constantly in training loops (`if epoch % checkpoint_every == 0:`, `if val_loss < best_val_loss:`).

In [ ]:
for item in d.items():
    print(item)

('Name', 'Angela Merkel')
('Country', 'Germany')
('Profession', 'Chancelor')
('Age', 65)


Iterating `d.items()` yields **4 tuples**, one per key-value pair. Order matches insertion order (with Age now 65 after the birthday increment). Note this is **guaranteed in Python 3.7+** -- prior to that it was an implementation detail of CPython 3.6.

**Practical consequence:** dictionaries are now the standard for ordered key-value data, replacing many uses of `collections.OrderedDict`. JSON serialization (`json.dumps`) preserves insertion order, which matters for reproducible config logging.

In [ ]:
for value in d.values():
    print(type(value))

<class 'str'>
<class 'str'>
<class 'str'>
<class 'int'>


The 4 values have types `str, str, str, int`. **Heterogeneous values** are typical in real configs: `{'lr': 1e-3, 'optimizer': 'adam', 'use_dropout': True}` mixes `float`, `str`, `bool`. This is one reason dicts are preferred over arrays for configuration -- arrays demand a single type.

### 3.2.4 Sets

A `set` is a **mutable, unordered, hashed collection of unique elements**. Set theory underlies many ML operations: vocabulary construction, label space management, deduplication, and graph algorithms.

In [ ]:
s = set(['u', 'd', 'ud', 'du', 'd', 'du'])
s

{'d', 'du', 'u', 'ud'}

The input list has **6 elements with 2 duplicates** (`'d'` appears twice, `'du'` appears twice). The resulting set has only **4 unique elements**: `{'d', 'u', 'ud', 'du'}`. Construction cost is $O(n)$, where $n$ is the size of the input.

**ML application:** building a vocabulary from raw tokens is precisely this operation:

```python
vocab = set(token for doc in corpus for token in doc.split())
```

For a 25,000-document corpus with 137 average tokens each, this scans $\approx 3.4M$ tokens but produces only the unique types -- typically 50-200K for English, depending on preprocessing aggressiveness.

In [ ]:
tset = set(['d', 'dd', 'uu', 'u'])

s.union(tset)

{'d', 'dd', 'du', 'u', 'ud', 'uu'}

**Union** $A \cup B$ contains every element in $A$ *or* $B$:

$$s \cup t = \{\text{d}, \text{u}, \text{ud}, \text{du}\} \cup \{\text{d}, \text{dd}, \text{uu}, \text{u}\} = \{\text{d}, \text{u}, \text{ud}, \text{du}, \text{dd}, \text{uu}\}$$

The result has **6 unique elements** -- four from `s`, two new ones from `tset` (`'dd'` and `'uu'`), and the shared `'d'` and `'u'` deduplicated.

In [ ]:
s.intersection(tset)

{'d', 'u'}

**Intersection** $A \cap B$ contains elements in *both*:

$$s \cap t = \{\text{d}, \text{u}\}$$

Only **2 elements** are shared. Intersection is the foundation of the **Jaccard similarity** between two sets:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{2}{6} = 0.33$$

Jaccard is a fundamental metric in NLP (text similarity), recommender systems (user-item co-occurrence), and clustering evaluation.

In [ ]:
s.difference(tset)

{'du', 'ud'}

**Difference** $A \setminus B$ contains elements of $A$ that are not in $B$:

$$s \setminus t = \{\text{ud}, \text{du}\}$$

These are the **2 elements** unique to `s`. Difference is asymmetric: `s.difference(tset) ≠ tset.difference(s)`.

In [ ]:
tset.difference(s)

{'dd', 'uu'}

The **reverse difference** $t \setminus s = \{\text{dd}, \text{uu}\}$ -- the **2 elements** unique to `tset`. Together with the previous result, the four "asymmetric" elements partition into `s`-only and `t`-only halves.

In [ ]:
s.symmetric_difference(tset)

{'dd', 'du', 'ud', 'uu'}

**Symmetric difference** $A \triangle B$ contains elements in *exactly one* of $A$ or $B$ (the XOR of set membership):

$$s \triangle t = (s \setminus t) \cup (t \setminus s) = \{\text{ud}, \text{du}, \text{dd}, \text{uu}\}$$

The result has **4 elements**. By the inclusion-exclusion principle:

$$|A \triangle B| = |A| + |B| - 2|A \cap B| = 4 + 4 - 2 \cdot 2 = 4 \;\checkmark$$

**ML application:** symmetric difference quantifies the disagreement between two label sets. If $A$ is the set of articles a model predicts as positive and $B$ is the ground-truth positive set, then $|A \triangle B|$ is the *total error count* (false positives + false negatives), and Jaccard is $1 - \frac{|A \triangle B|}{|A \cup B|}$.

#### Application: deduplication

The most common use of `set` in everyday data work is removing duplicates from a collection.

In [ ]:
seed(0)
ll = [randint(0, 10) for i in range(1000)]
len(ll)

1000

We generate **1,000 random integers** uniformly from $\{0, 1, ..., 10\}$. With a seeded PRNG (`seed(0)`), the sequence is deterministic across runs. **Reproducibility is non-negotiable in ML research** -- every paper, every benchmark, every CI pipeline depends on seeds being fixed.

In [ ]:
ll[:20]

[6, 6, 0, 4, 8, 7, 6, 4, 7, 5, 9, 3, 8, 2, 4, 2, 1, 9, 4, 8]

The first 20 elements illustrate the duplication: `6` appears 3 times, `4` appears 4 times. Across the full 1,000-element list, the **expected number of distinct values** is the number of distinct outcomes the PRNG can produce -- which is exactly 11 (the integers 0 through 10), since the *coupon collector* expected count for drawing all 11 values is $11 \sum_{k=1}^{11} \frac{1}{k} \approx 33$ draws -- vastly fewer than our 1,000.

In [ ]:
set(ll)

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}

Converting to a set yields **exactly 11 unique values**: $\{0, 1, 2, ..., 10\}$. With 1,000 draws from 11 buckets, the average bucket count is $\frac{1000}{11} \approx 91$ -- so each integer appears on average 91 times. The empirical compression ratio is:

$$\text{ratio} = \frac{|\text{set}(ll)|}{|ll|} = \frac{11}{1000} = 1.1\%$$

Storage shrinks by **~99%** when duplicates are eliminated.

**Production lesson:** in real datasets, deduplication is a critical preprocessing step. Repeated training samples cause:

1. **Inflated effective epoch count** -- the model sees the same example multiple times per epoch.
2. **Train-test leakage** -- if a duplicate appears in both splits, the model's test "score" is partly memorization.
3. **Biased class distribution** -- duplicates are rarely uniform across classes.

The first thing to do with any dataset is `len(df) - len(df.drop_duplicates())`. If that number is nonzero, investigate.

## Conclusion

This chapter introduced Python's basic data types and structures, the atomic vocabulary on which every later chapter builds. Stepping back, we can summarize the key takeaways through an ML/DS lens:

**Atomic types** -- `int`, `float`, `bool`, `str` -- map directly onto:

| Python type | ML/DS counterpart | Key concern |
|---|---|---|
| `int` | Token IDs, indices, sample counts | Overflow rare in pure Python; common in `int32` tensors |
| `float` | Weights, losses, probabilities | IEEE 754 rounding (~16 digits); use `decimal.Decimal` for finance |
| `bool` | Masks, indicators, labels | $\mathbb{1}[\cdot]$ is the bridge to vectorized accuracy / F1 |
| `str` | Tokens, log entries, paths | Tokenization, regex, and encoding form the NLP entry point |

**Container types** -- `tuple`, `list`, `dict`, `set` -- map onto:

| Python type | ML/DS counterpart | Key property |
|---|---|---|
| `tuple` | Tensor shapes, return records, hash keys | Immutable, hashable, $O(1)$ access |
| `list` | Mini-batches, training history, samples | Mutable, $O(1)$ amortized append, $O(n)$ insert |
| `dict` | Hyperparameter configs, JSON payloads, feature stores | $O(1)$ average key lookup, insertion-ordered (3.7+) |
| `set` | Vocabularies, label spaces, dedup | Unique-element collection, $O(1)$ membership |

**Three foundational principles** to carry into Chapter 4 and beyond:

1. **Floating-point arithmetic is approximate.** Never use `==` on floats. The ~16-digit precision of `float64` is the upper bound on the accuracy of any standard ML computation.
2. **Vectorize whenever possible.** List comprehensions, `map`, and `filter` are stepping stones; NumPy's `ndarray` (next chapter) is the destination. The 100x speedup from vectorization is what makes modern ML practical.
3. **Choose containers by access pattern.** Need ordered, mutable, indexable? `list`. Need O(1) lookup by key? `dict`. Need uniqueness? `set`. Need immutability and hashability? `tuple`. The right choice can mean the difference between $O(n)$ and $O(n^2)$ in production code.

In **Chapter 4** we move from these scalar/collection primitives to **NumPy arrays**, where the `tuple` becomes the shape descriptor and the `float` becomes a contiguous block of bytes -- delivering the 10-100x performance gains that make data science at scale possible.